# A/B Test Analysis: Comparing New Features vs Legacy

This notebook provides automated comparison between control (legacy)
and treatment (new feature) experiment runs.

**Prerequisites**: Run each A/B pair using the configs in `examples/ab_test_*.yaml`.
Edit the toggle field in each YAML to produce the B-group run.

In [ ]:
import json
import re
from collections import Counter, defaultdict
from data_loader import (
    load_experiment,
    extract_action_events,
    extract_movement_events,
    extract_reflection_events,
    compute_location_transition_matrix,
    compute_interaction_matrix,
)

from typing import Any

## Configuration

Fill in the experiment IDs for each A/B pair. Each pair should use
the same random_seed and simulation_steps, differing only in the
tested feature.

In [ ]:
ab_pairs = {
    "Reflection": {
        "control": "ab_reflect_off",       # reflection_enabled: false
        "treatment": "ab_reflect_on",       # reflection_enabled: true
    },
    "FOV (basic)": {
        "control": "ab_fov_off",            # fov_enabled: false
        "treatment": "ab_fov_on",            # fov_enabled: true
    },
    "FOV (extended)": {
        "control": "ab_fov_ext_off",        # fov_enabled: false
        "treatment": "ab_fov_ext_on",        # fov_enabled: true, distance=2.0
    },
    "Path Planner": {
        "control": "ab_path_legacy",        # path_planner_enabled: false
        "treatment": "ab_path_astar",        # path_planner_enabled: true
    },
    "Goal Planning": {
        "control": "ab_goal_off",           # goal_enabled: false
        "treatment": "ab_goal_on",           # goal_enabled: true
    },
    "Spatial Topology": {
        "control": "ab_ring",               # no spatial_config (ring)
        "treatment": "ab_smallworld",       # spatial_config: small_world
    },
}

## Utility: Load and Compare a Single A/B Pair

In [ ]:
def load_pair(pair_name: str, eid_a: str, eid_b: str) -> tuple[dict, dict]:
    """Load events for an A/B pair."""
    events_a = load_experiment(eid_a)
    events_b = load_experiment(eid_b)
    summary_a = {"id": eid_a, "total": len(events_a)}
    summary_b = {"id": eid_b, "total": len(events_b)}
    if events_a:
        steps_a = [e.get("step", 0) for e in events_a]
        summary_a["steps"] = max(steps_a)
    if events_b:
        steps_b = [e.get("step", 0) for e in events_b]
        summary_b["steps"] = max(steps_b)
    return events_a, events_b

In [ ]:
def metric_action_diversity(events: list[dict]) -> float:
    """Ratio of unique actions to total actions. Higher = more diverse."""
    actions = extract_action_events(events)
    if not actions:
        return 0.0
    texts = []
    for a in actions:
        data = a.get("data", {})
        if isinstance(data, dict):
            texts.append(data.get("action", "").strip())
        elif isinstance(data, str):
            texts.append(data.strip())
    texts = [t for t in texts if t]
    if not texts:
        return 0.0
    return len(set(texts)) / len(texts)

In [ ]:
def metric_interaction_count(events: list[dict]) -> int:
    """Total agent-to-agent interaction mentions."""
    matrix = compute_interaction_matrix(events)
    return sum(sum(v.values()) for v in matrix.values())

In [ ]:
def metric_movement_diversity(events: list[dict]) -> float:
    """Ratio of unique destination locations to total moves."""
    movements = extract_movement_events(events)
    if not movements:
        return 0.0
    destinations = []
    for m in movements:
        data = m.get("data", {})
        if isinstance(data, dict):
            d = data.get("to", "")
            if d:
                destinations.append(d)
    if not destinations:
        return 0.0
    return len(set(destinations)) / len(destinations)

In [ ]:
def metric_reflection_count(events: list[dict]) -> int:
    """Number of reflection events."""
    return len(extract_reflection_events(events))

In [ ]:
def metric_transition_entropy(events: list[dict]) -> float:
    """Shannon entropy of location transitions. Higher = more spread movement.
    
    Measures how uniformly agents distribute their movement across
    all possible transitions. Low entropy means agents favor a few routes.
    """
    import math
    tm = compute_location_transition_matrix(events)
    total = sum(sum(v.values()) for v in tm.values())
    if total == 0:
        return 0.0
    entropy = 0.0
    for src, dests in tm.items():
        for dst, count in dests.items():
            p = count / total
            if p > 0:
                entropy -= p * math.log(p)
    return entropy

In [ ]:
def metric_goal_review_count(events: list[dict]) -> int:
    """Number of goal_review events (only in goal-enabled runs)."""
    return len([e for e in events if e.get("event_type") == "goal_review"])

In [ ]:
def metric_daily_plan_consistency(events: list[dict]) -> float:
    """Jaccard similarity between daily plans across days.
    
    1.0 = identical plans every day, 0.0 = completely different.
    """
    daily_plans = []
    for e in events:
        if e.get("event_type") == "daily_plan":
            data = e.get("data", {})
            if isinstance(data, dict):
                text = data.get("plan", "").strip()
            elif isinstance(data, str):
                text = data.strip()
            else:
                text = ""
            if text:
                daily_plans.append(set(text.lower().split()))
    
    if len(daily_plans) < 2:
        return 0.0
    
    similarities = []
    for i in range(len(daily_plans) - 1):
        a, b = daily_plans[i], daily_plans[i + 1]
        if a or b:
            intersection = len(a & b)
            union = len(a | b)
            similarities.append(intersection / union if union else 0.0)
    
    return sum(similarities) / len(similarities) if similarities else 0.0

## Run All A/B Comparisons

In [ ]:
# Metric definitions per A/B pair
# Each pair specifies which metrics are relevant and the expected direction
metric_defs = {
    "Reflection": [
        ("Action Diversity", metric_action_diversity, "higher_is_better"),
        ("Reflection Count", metric_reflection_count, "higher_is_better"),
        ("Daily Plan Consistency", metric_daily_plan_consistency, "higher_is_better"),
    ],
    "FOV (basic)": [
        ("Action Diversity", metric_action_diversity, "neutral"),
        ("Interaction Count", metric_interaction_count, "higher_is_better"),
        ("Movement Diversity", metric_movement_diversity, "neutral"),
    ],
    "FOV (extended)": [
        ("Action Diversity", metric_action_diversity, "neutral"),
        ("Interaction Count", metric_interaction_count, "higher_is_better"),
        ("Movement Diversity", metric_movement_diversity, "neutral"),
    ],
    "Path Planner": [
        ("Movement Diversity", metric_movement_diversity, "neutral"),
        ("Action Diversity", metric_action_diversity, "neutral"),
        ("Transition Entropy", metric_transition_entropy, "neutral"),
    ],
    "Goal Planning": [
        ("Action Diversity", metric_action_diversity, "neutral"),
        ("Daily Plan Consistency", metric_daily_plan_consistency, "higher_is_better"),
        ("Goal Review Count", metric_goal_review_count, "higher_is_better"),
    ],
    "Spatial Topology": [
        ("Movement Diversity", metric_movement_diversity, "neutral"),
        ("Transition Entropy", metric_transition_entropy, "neutral"),
        ("Interaction Count", metric_interaction_count, "neutral"),
    ],
}

In [ ]:
def run_comparison(pair_name: str, control_id: str, treatment_id: str) -> None:
    """Run A/B comparison for a single pair."""
    print(f"\n{'='*60}")
    print(f"  {pair_name}")
    print(f"{'='*60}")
    
    try:
        events_a, events_b = load_pair(pair_name, control_id, treatment_id)
    except FileNotFoundError as e:
        print(f"  SKIP: {e}")
        return
    
    metrics = metric_defs.get(pair_name, [])
    if not metrics:
        print(f"  No metrics defined for this pair.")
        return
    
    print(f"  {'Metric':<25} {'Control':>10} {'Treatment':>10} {'Delta':>10} {'Direction':>15}")
    print(f"  {'-'*70}")
    
    significant = 0
    for metric_name, metric_fn, direction in metrics:
        val_a = metric_fn(events_a)
        val_b = metric_fn(events_b)
        delta = val_b - val_a
        pct = (delta / val_a * 100) if val_a != 0 else float('inf')
        
        arrow = ""
        if direction == "higher_is_better" and delta > 0:
            arrow = " <<<"
            significant += 1
        elif direction == "higher_is_better" and delta < 0:
            arrow = " (no effect)"
        elif direction == "neutral":
            if abs(delta) > 0.05:
                arrow = " (different)"
            else:
                arrow = " (similar)"
        
        pct_str = f"{pct:+.1f}%" if abs(pct) < 1000 else "N/A"
        print(f"  {metric_name:<25} {val_a:>10.3f} {val_b:>10.3f} {pct_str:>10} {arrow}")
    
    total = len(metrics)
    print(f"\n  Result: {significant}/{total} metrics show expected improvement.")
    if significant == total and total > 0:
        print(f"  VERDICT: Feature shows clear positive impact.")
    elif significant > 0:
        print(f"  VERDICT: Feature shows partial improvement. Review individual metrics.")
    else:
        print(f"  VERDICT: No significant improvement detected. Feature may need tuning.")

## Execute All Pairs

In [ ]:
results = {}
for pair_name, pair in ab_pairs.items():
    run_comparison(
        pair_name,
        pair["control"],
        pair["treatment"],
    )
    print()

## Summary Table

In [ ]:
# Re-generate summary for all pairs
print("\n" + "=" * 75)
print("A/B TEST SUMMARY")
print("=" * 75)

for pair_name, pair in ab_pairs.items():
    metrics = metric_defs.get(pair_name, [])
    if not metrics:
        continue
    
    try:
        events_a, events_b = load_pair(pair_name, pair["control"], pair["treatment"])
    except FileNotFoundError:
        continue
    
    # Compute all metrics
    vals_a = [m[1](events_a) for m in metrics]
    vals_b = [m[1](events_b) for m in metrics]
    
    print(f"\n{pair_name}:")
    for (name, _, direction), va, vb in zip(metrics, vals_a, vals_b):
        delta = vb - va
        sign = "+" if delta >= 0 else ""
        print(f"  {name}: {sign}{delta:.3f}")
    print()

## Interpretation Guide

| Metric | High Value Means | How to Interpret |
|--------|-----------------|-------------------|
| Action Diversity | Agents do different things | Treatment broadened behavior? |
| Interaction Count | Agents interact more | Treatment enabled social awareness? |
| Movement Diversity | Agents visit more places | Treatment increased exploration? |
| Reflection Count | Agents reflect more | Reflection system is active? |
| Daily Plan Consistency | Plans are similar across days | Goals create persistent focus? |
| Transition Entropy | Movement is spread evenly | Topology affects path choices? |
| Goal Review Count | Goals are reviewed | Goal system is active? |

A positive delta (B > A) for `higher_is_better` metrics indicates the
feature improves that aspect. For `neutral` metrics, large deltas
indicate the feature changes behavior significantly (not necessarily better).

**Note**: With LLM-driven agents, some variance is expected across runs
even with the same seed. Use batch seeds (3+) for statistical confidence.